<a href="https://colab.research.google.com/github/TirthankaSaha/AI-powered-Virtual-Development-Pod/blob/main/SDLC_Pod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install required packages
!pip install openai==0.28 gradio PyMuPDF fpdf faiss-cpu sentence-transformers --quiet

# Imports
import gradio as gr
import fitz  # PyMuPDF
import openai
import subprocess
import tempfile
import os
import torch
import faiss
from fpdf import FPDF
from sentence_transformers import SentenceTransformer

# OpenRouter API setup
openai.api_key = "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"  # Replace with your actual OpenRouter key
openai.api_base = "https://openrouter.ai/api/v1"

# Session state
session_state = {
    "user_stories": None,
    "design_doc": None,
    "generated_code": None,
    "test_cases": None,
    "rfp_chunks": [],
    "chunk_embeddings": None,
    "index": None
}

# Load predefined format templates
with open("business_analyst_format.txt") as f:
    BUSINESS_ANALYST_FORMAT = f.read()
with open("design_engineer_format.txt") as f:
    DESIGN_ENGINEER_FORMAT = f.read()
with open("developer_format.txt") as f:
    DEVELOPER_FORMAT = f.read()
with open("qa_tester_format.txt") as f:
    QA_TESTER_FORMAT = f.read()

# RAG Setup
model = SentenceTransformer("all-MiniLM-L6-v2")

def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file.name)
    return "".join([page.get_text() for page in doc])

def split_text(text, chunk_size=300, overlap=50):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size-overlap)]

def store_pdf_embeddings(chunks):
    embeddings = model.encode(chunks, convert_to_tensor=True)
    session_state["rfp_chunks"] = chunks
    session_state["chunk_embeddings"] = embeddings.cpu().detach().numpy()
    index = faiss.IndexFlatL2(session_state["chunk_embeddings"].shape[1])
    index.add(session_state["chunk_embeddings"])
    session_state["index"] = index

def retrieve_relevant_chunks(query, top_k=5):
    query_embedding = model.encode([query], convert_to_tensor=True).cpu().detach().numpy()
    D, I = session_state["index"].search(query_embedding, top_k)
    return [session_state["rfp_chunks"][i] for i in I[0]]

def export_to_pdf():
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    sections = {
        "User Stories": session_state.get("user_stories", "No user stories generated."),
        "System Design": session_state.get("design_doc", "No design document generated."),
        "Backend Code": session_state.get("generated_code", "No code generated."),
        "Test Cases": session_state.get("test_cases", "No test cases generated.")
    }
    for title, content in sections.items():
        pdf.set_font("Arial", 'B', 14)
        pdf.cell(200, 10, txt=title, ln=True)
        pdf.set_font("Arial", size=10)
        for line in content.split("\n"):
            pdf.multi_cell(0, 5, txt=line)
        pdf.ln(5)
    temp_file_path = os.path.join(tempfile.gettempdir(), "virtual_dev_pod_output.pdf")
    pdf.output(temp_file_path)
    return temp_file_path

def format_prompt_with_context(role_description, format_template, context):
    return f"""
You are a {role_description} AI agent.

Relevant RFP Context:
{context}

Format:
{format_template}

Output:
"""

def business_analyst_agent(rfp_pdf):
    try:
        text = extract_text_from_pdf(rfp_pdf)
        chunks = split_text(text)
        store_pdf_embeddings(chunks)
        matched_chunks = retrieve_relevant_chunks("Generate user stories based on the RFP.")
        context = "\n".join(matched_chunks)
        prompt = format_prompt_with_context("Business Analyst", BUSINESS_ANALYST_FORMAT, context)
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[
                {"role": "system", "content": "You are an expert business analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        session_state["user_stories"] = response['choices'][0]['message']['content']
        return session_state["user_stories"]
    except Exception as e:
        return f"❌ Error: {str(e)}"

def design_agent(_):
    try:
        prompt = f"""
You are a Software Design AI.

User Stories:
{session_state['user_stories']}

Format:
{DESIGN_ENGINEER_FORMAT}
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7
        )
        session_state["design_doc"] = response['choices'][0]['message']['content']
        return session_state["design_doc"]
    except Exception as e:
        return f"❌ Error in Design Agent: {str(e)}"

def code_agent(_):
    try:
        prompt = f"""
You are a Backend Developer AI.

User Stories:
{session_state['user_stories']}

Design Doc:
{session_state['design_doc']}

Format:
{DEVELOPER_FORMAT}
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.6
        )
        session_state["generated_code"] = response['choices'][0]['message']['content']
        return session_state["generated_code"]
    except Exception as e:
        return f"❌ Error in Code Agent: {str(e)}"

def testing_agent(_):
    try:
        prompt = f"""
You are a QA Testing AI.

User Stories:
{session_state['user_stories']}

Code:
{session_state['generated_code']}

Format:
{QA_TESTER_FORMAT}
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.5
        )
        session_state["test_cases"] = response['choices'][0]['message']['content']
        return session_state["test_cases"]
    except Exception as e:
        return f"❌ Error in Testing Agent: {str(e)}"

def run_tests(_):
    try:
        with tempfile.TemporaryDirectory() as tmpdirname:
            code_path = os.path.join(tmpdirname, "app.py")
            test_path = os.path.join(tmpdirname, "test_app.py")
            with open(code_path, "w") as f:
                f.write(session_state["generated_code"])
            with open(test_path, "w") as f:
                f.write(session_state["test_cases"])
            result = subprocess.run(
                ["pytest", test_path, "--tb=short", "-q"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                cwd=tmpdirname
            )
            return f"✅ Test Results:\n\n{result.stdout}"
    except Exception as e:
        return f"❌ Error while running tests: {str(e)}"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🤖 NEXUS : AI-Powered Virtual Development Pod

    🚀 This virtual pod simulates a complete software development workflow using AI agents:
    - 🧠 Business Analyst (User Stories)
    - 🏗️ Design Engineer (System Design)
    - 👨‍💻 Developer (Backend Code)
    - 🧪 QA Tester (Test Cases)
    - ✅ Test Runner (Test Cases Validation)
    - 📄 Export Output to PDF

    Upload your RFP PDF and let the agents handle the rest!
    """)

    with gr.Tabs():
        with gr.Tab("1️⃣ Business Analyst Agent"):
            rfp_input = gr.File(label="📄 Upload RFP PDF", file_types=[".pdf"])
            submit_btn = gr.Button("✨ Generate User Stories")
            output = gr.Textbox(label="📋 Generated User Stories", lines=20)
            submit_btn.click(fn=business_analyst_agent, inputs=[rfp_input], outputs=[output])

        with gr.Tab("2️⃣ Design Agent"):
            design_btn = gr.Button("🛠️ Generate Design Document")
            design_output = gr.Textbox(label="📐 Generated Design Document", lines=20)
            design_btn.click(fn=design_agent, inputs=[], outputs=[design_output])

        with gr.Tab("3️⃣ Coding Agent"):
            code_btn = gr.Button("💻 Generate Backend Code")
            code_output = gr.Code(label="🧩 Generated Backend Code", language="python", lines=20)
            code_btn.click(fn=code_agent, inputs=[], outputs=[code_output])

        with gr.Tab("4️⃣ Testing Agent"):
            test_btn = gr.Button("🔍 Generate Test Cases")
            test_output = gr.Code(label="🧪 Generated Test Cases", language="python", lines=20)
            test_btn.click(fn=testing_agent, inputs=[], outputs=[test_output])

        with gr.Tab("5️⃣ Test Runner"):
            run_test_btn = gr.Button("🏁 Run Test Cases")
            test_result_output = gr.Textbox(label="✅ Test Cases Run Output", lines=20)
            run_test_btn.click(fn=run_tests, inputs=[], outputs=[test_result_output])

        with gr.Tab("6️⃣ Summary PDF"):
            export_btn = gr.Button("📤 Export All to PDF")
            pdf_file_output = gr.File(label="📄 Download PDF")
            export_btn.click(fn=export_to_pdf, inputs=[], outputs=[pdf_file_output])

demo.launch()


/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function design_agent at 0x7ded6ce8e3e0>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expected at least 1 arguments for function <function design_agent at 0x7ded6ce8e3e0>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function code_agent at 0x7ded6ce8e200>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expected at least 1 arguments for function <function code_agent at 0x7ded6ce8e200>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function testing_agent at 0x7ded6ce8e840>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expec

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7b9a99979dab0e031.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
